In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K

input_dim = 20
latent_dim = 2

# Encoder
inputs = Input(shape=(input_dim,))
h = Dense(10, activation='relu')(inputs)

z_mean = Dense(latent_dim)(h)
z_log_var = Dense(latent_dim)(h)

# Sampling function
def sampling(args):
    z_mean, z_log_var = args
    epsilon = K.random_normal(shape=(K.shape(z_mean)[0], latent_dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling)([z_mean, z_log_var])

# Decoder
decoder_h = Dense(10, activation='relu')
decoder_out = Dense(input_dim, activation='sigmoid')

h_decoded = decoder_h(z)
outputs = decoder_out(h_decoded)

vae = Model(inputs, outputs)

# Loss (reconstruction + KL divergence)
reconstruction_loss = tf.keras.losses.mse(inputs, outputs)
kl_loss = -0.5 * K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1)
vae_loss = K.mean(reconstruction_loss + kl_loss)

vae.add_loss(vae_loss)
vae.compile(optimizer='adam')

# Dummy training data
import numpy as np
x_train = np.random.rand(1000, input_dim)

vae.fit(x_train, epochs=5, batch_size=32)